# SentinelMail — Data Exploration: ai4privacy/pii-masking-300k

Exploratory analysis of the primary training dataset before any model training.
Label mapping and privacy rules follow `.claude/rules/datasets.md`.

In [8]:
import json
import os
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import load_dataset

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

REPO_ROOT = Path("..")
DATA_RAW = REPO_ROOT / "data" / "ai4privacy" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "ai4privacy" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# FinPII entity types (finance/insurance subset per datasets.md)
FINPII_TYPES = {
    "ACCOUNTNUMBER", "CREDITCARDNUMBER", "CREDITCARDCVV", "CREDITCARDISSUER",
    "IBAN", "SWIFT", "CURRENCY", "CURRENCYNAME", "CURRENCYCODE", "CURRENCYSYMBOL",
    "AMOUNT", "TAXNUMBER", "SOCIALSECURITYNUMBER", "SORTCODE", "ETHEREUMADDRESS",
    "BITCOINADDRESS", "LITECOINADDRESS",
}

## 1. Download Dataset

In [9]:
ds = load_dataset("ai4privacy/pii-masking-300k", cache_dir=str(DATA_RAW))
print(ds)

DatasetDict({
    train: Dataset({
        features: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set'],
        num_rows: 177677
    })
    validation: Dataset({
        features: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set'],
        num_rows: 47728
    })
})


## 2. Dataset Overview

In [10]:
print("Splits:", list(ds.keys()))
for split, dataset in ds.items():
    print(f"  {split}: {len(dataset):,} rows | columns: {dataset.column_names}")

# Show one sample — truncate source_text to 50 chars per privacy rule
sample = ds["train"][0]
sample_display = {k: (v[:50] + "\u2026" if k == "source_text" and isinstance(v, str) else v)
                  for k, v in sample.items()}
print("\nSample row (source_text truncated to 50 chars):")
for k, v in sample_display.items():
    print(f"  {k}: {v}")

Splits: ['train', 'validation']
  train: 177,677 rows | columns: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set']
  validation: 47,728 rows | columns: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set']

Sample row (source_text truncated to 50 chars):
  source_text: Subject: Group Messaging for Admissions Process

G…
  target_text: Subject: Group Messaging for Admissions Process

Good morning, everyone,

I hope this message finds you well. As we continue our admissions processes, I would like to update you on the latest developments and key information. Please find below the timeline for our upcoming meetings:

- [USERNAME] - Meeting at [TIME]
- [USERNAME] - Meeting at [TIME]
- [USERNAME] - Meeting at [TIME]
- [USERNAME] - Meeting at [TIME]
- [USERNAME] 
  privacy_mask: [{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USE

## 3. Filter to English

In [11]:
# Dataset uses full names ("English"), not ISO codes ("en")
ds_en = ds.filter(lambda x: x["language"] == "English")

print("Rows kept after English filter:")
for split in ds_en:
    orig = len(ds[split])
    kept = len(ds_en[split])
    print(f"  {split}: {kept:,} / {orig:,} ({100*kept/orig:.1f}%)")

Filter: 100%|██████████| 47728/47728 [00:04<00:00, 9996.36 examples/s] 

Rows kept after English filter:
  train: 29,908 / 177,677 (16.8%)
  validation: 7,946 / 47,728 (16.6%)


## 4. Column Schema

In [ ]:
train_df = ds_en["train"].to_pandas()

print("Column dtypes:")
print(train_df.dtypes)
print("\nNull counts:")
print(train_df.isnull().sum())

# Debug: show actual type/value of privacy_mask before filtering
print("\nprivacy_mask — first 3 raw values:")
for i, v in enumerate(train_df["privacy_mask"].head(3)):
    print(f"  [{i}] type={type(v).__name__}  value={repr(v)[:120]}")

def _has_entities(x):
    if x is None:
        return False
    if isinstance(x, float):  # catches pandas NA / NaN
        return False
    # HuggingFace → pandas often yields numpy.ndarray, not list
    if isinstance(x, (list, np.ndarray)):
        return len(x) > 0
    if isinstance(x, str):
        try:
            return len(json.loads(x)) > 0
        except (json.JSONDecodeError, TypeError):
            return False
    return False

mask_rows = train_df[train_df["privacy_mask"].apply(_has_entities)]
print(f"\nRows with non-empty privacy_mask: {len(mask_rows):,}")

if len(mask_rows) > 0:
    mask_val = mask_rows.iloc[0]["privacy_mask"]
    if isinstance(mask_val, str):
        mask_val = json.loads(mask_val)
    print("\nprivacy_mask schema example:")
    print(json.dumps(mask_val[:2], indent=2, default=str))
else:
    print("\nWARNING: no non-empty privacy_mask rows found — check column name/format above.")

## 5. PII Entity Class Distribution

In [ ]:
def parse_mask(mask):
    """Return a list of entity dicts from privacy_mask regardless of storage format."""
    if mask is None or (isinstance(mask, float)):  # None or NaN
        return []
    # HuggingFace → pandas often yields numpy.ndarray, not list
    if isinstance(mask, np.ndarray):
        return list(mask)
    if isinstance(mask, list):
        return mask
    if isinstance(mask, str):
        try:
            return json.loads(mask)
        except (json.JSONDecodeError, TypeError):
            return []
    return []

entity_counts = Counter()
for mask in train_df["privacy_mask"]:
    for ent in parse_mask(mask):
        entity_counts[ent["label"]] += 1

if not entity_counts:
    print("No entities found — check privacy_mask format in cell 4 output.")
else:
    top20 = entity_counts.most_common(20)
    labels_top, counts_top = zip(*top20)

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(labels_top[::-1], counts_top[::-1])
    ax.set_xlabel("Entity occurrences (train, EN)")
    ax.set_title("Top 20 PII entity types in ai4privacy (English train split)")
    for bar, count in zip(bars, counts_top[::-1]):
        ax.text(bar.get_width() + max(counts_top) * 0.005, bar.get_y() + bar.get_height() / 2,
                f"{count:,}", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()

    print(f"\nTotal entity types in dataset: {len(entity_counts)}")
    print(f"Total entity occurrences: {sum(entity_counts.values()):,}")

## 6. FinPII Subset

In [ ]:
def has_finpii(mask):
    return any(e["label"] in FINPII_TYPES for e in parse_mask(mask))

if len(train_df) == 0:
    raise ValueError(
        "train_df is empty — re-run the English filter cell "
        '(language must be "English", not "en").'
    )

finpii_mask = train_df["privacy_mask"].apply(has_finpii)
n_finpii = finpii_mask.sum()
print(f"Rows with ≥1 FinPII entity: {n_finpii:,} ({100*n_finpii/len(train_df):.1f}% of EN train)")

finpii_entity_counts = Counter(
    e["label"]
    for mask in train_df["privacy_mask"]
    for e in parse_mask(mask)
    if e["label"] in FINPII_TYPES
)
print("\nFinPII entity distribution:")
for label, cnt in finpii_entity_counts.most_common():
    print(f"  {label}: {cnt:,}")


## 7. Label Mapping (Project Schema)

In [ ]:
def map_labels(mask):
    entities = parse_mask(mask)
    has_any_pii = len(entities) > 0
    has_fin = any(e["label"] in FINPII_TYPES for e in entities)
    return {
        "PII": int(has_any_pii),
        "financial": int(has_fin),
        "benign": int(not has_any_pii),
        "health": 0,
        "confidential": 0,
    }

label_df = train_df["privacy_mask"].apply(map_labels).apply(pd.Series)
train_df = pd.concat([train_df, label_df], axis=1)

print("Label distribution (train, EN):")
print(label_df[["PII", "financial", "benign", "health", "confidential"]].sum().to_string())

fig, ax = plt.subplots(figsize=(8, 4))
counts = label_df[["benign", "PII", "financial"]].sum()
bars = ax.bar(counts.index, counts.values, color=sns.color_palette("muted", 3))
ax.set_ylabel("Number of examples")
ax.set_title("Class distribution after label mapping (train, EN)")
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
            f"{val:,}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

print("\nLabel co-occurrence (financial \u2229 PII must equal financial count):")
co = label_df[["PII", "financial", "benign"]].T.dot(label_df[["PII", "financial", "benign"]])
print(co)

## 8. Text Length Distribution

In [ ]:
train_df["token_count"] = train_df["mbert_text_tokens"].apply(
    lambda x: len(x) if isinstance(x, (list, np.ndarray)) else len(json.loads(x))
)

over_512 = (train_df["token_count"] > 512).sum()
print(f"Examples exceeding 512 tokens: {over_512:,} ({100*over_512/len(train_df):.1f}%)")
print(train_df["token_count"].describe())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(train_df["token_count"].clip(upper=600), bins=60, edgecolor="white")
ax.axvline(512, color="red", linestyle="--", linewidth=1.5, label="512-token limit (BERT)")
ax.set_xlabel("Token count (mbert_text_tokens, clipped at 600)")
ax.set_ylabel("Examples")
ax.set_title("Text length distribution (train, EN)")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Entity Count per Example

In [ ]:
train_df["num_entities"] = train_df["privacy_mask"].apply(lambda m: len(parse_mask(m)))

print("Entity count distribution:")
print(train_df["num_entities"].value_counts().sort_index().head(15).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

vc = train_df["num_entities"].value_counts().sort_index()
axes[0].bar(vc.index[:15], vc.values[:15])
axes[0].set_xlabel("Number of entities per example")
axes[0].set_ylabel("Count")
axes[0].set_title("Entity count distribution (0\u201314)")

sns.boxplot(data=train_df[train_df["num_entities"] > 0], y="num_entities", ax=axes[1])
axes[1].set_title("Entity count distribution (examples with \u22651 entity)")
axes[1].set_ylabel("Number of entities")

plt.tight_layout()
plt.show()

## 10. Sample Inspection

In [ ]:
def show_samples(df, label_col, n=5):
    subset = df[df[label_col] == 1].head(n)
    for _, row in subset.iterrows():
        entities = [(e["label"], e["value"][:20]) for e in parse_mask(row["privacy_mask"])]
        print(f"  text[:50]: {row['source_text'][:50]}\u2026")
        print(f"  entities : {entities}")
        print()

print("=== PII-only examples ===")
show_samples(train_df[train_df["financial"] == 0], "PII")

print("=== Financial examples ===")
show_samples(train_df, "financial")

print("=== Benign examples ===")
show_samples(train_df, "benign")

## 11. Save Processed Label File

In [ ]:
# Save labels + metadata only — no source_text to avoid storing PII-like content
out_cols = ["token_count", "num_entities", "PII", "financial", "benign", "health", "confidential"]
out_df = train_df[out_cols].reset_index(names="idx")

out_path = DATA_PROCESSED / "labels_en.parquet"
out_df.to_parquet(out_path, index=False)
print(f"Saved {len(out_df):,} rows to {out_path}")
print(out_df.head())